# TT Means Test time trained
This notebook contains only the Qwen 2.5 0.5B model fine-tuning and inference. It is fine-tuned on the test data during test


**Note: This submission uses only the 0.5B Qwen model for standalone classification.**

Can be used to experiment how different models perform on Test-time training

# Original Notebook(1)
https://www.kaggle.com/code/kishanvavdara/test-on-testdataset-qwenemdding-llama-lr

Only the .5B model part was separated ,nothing else changed

In [ ]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

<cell_type>markdown</cell_type># Hybrid Model Strategy: 1.5B Test-Time Training + 32B Pre-trained

This notebook implements a split strategy:
- **1.5B model**: Test-time trained on non-legal/advertising rules
- **32B model**: Pre-trained model for legal/advertising rules only
- Final submission combines predictions from both models

In [ ]:
%%writefile constants.py
BASE_MODEL_PATH_1_5B = "/kaggle/input/qwen2.5/transformers/1.5b-instruct-gptq-int4/1"
BASE_MODEL_PATH_32B = "/kaggle/input/qwen2-5-32b-instruct-gptq-int4"
LORA_PATH_1_5B = "output/"
LORA_PATH_32B = "/kaggle/input/qwen2-5-32b-gptq-int4-batch4-full"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = '''You are given a comment from reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

def is_legal_or_advertising_rule(rule_text):
    rule_lower = rule_text.lower()
    return 'legal' in rule_lower or 'advertising' in rule_lower

In [ ]:
%%writefile utils.py
import pandas as pd
from datasets import Dataset
from constants import POSITIVE_ANSWER, NEGATIVE_ANSWER, COMPLETE_PHRASE, BASE_PROMPT, is_legal_or_advertising_rule
import random, numpy as np
random.seed(42)
np.random.seed(42)


def build_prompt(row):
    return f"""
{BASE_PROMPT}

Subreddit: r/{row["subreddit"]}
Rule: {row["rule"]}
Examples:
1) {row["positive_example"]}
{COMPLETE_PHRASE} Yes

2) {row["negative_example"]}
{COMPLETE_PHRASE} No

---
Comment: {row["body"]}
{COMPLETE_PHRASE}"""


def get_dataframe_to_train(data_path, exclude_legal_advertising=True):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv").reset_index(drop=True)

    if exclude_legal_advertising:
        train_dataset = train_dataset[~train_dataset['rule'].apply(is_legal_or_advertising_rule)].copy()
        test_dataset = test_dataset[~test_dataset['rule'].apply(is_legal_or_advertising_rule)].copy()

    flatten = []

    train_df = train_dataset[["body", "rule", "subreddit", "rule_violation",
                              "positive_example_1","positive_example_2",
                              "negative_example_1","negative_example_2"]].copy()

    train_df["positive_example"] = np.where(
        np.random.rand(len(train_df)) < 0.5,
        train_df["positive_example_1"],
        train_df["positive_example_2"]
    )
    train_df["negative_example"] = np.where(
        np.random.rand(len(train_df)) < 0.5,
        train_df["negative_example_1"],
        train_df["negative_example_2"]
    )

    train_df.drop(columns=["positive_example_1","positive_example_2",
                           "negative_example_1","negative_example_2"], inplace=True)

    flatten.append(train_df)

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[["rule","subreddit",
                                        "positive_example_1","positive_example_2",
                                        "negative_example_1","negative_example_2"]].copy()

            if violation_type == "positive":
                body_col = f"positive_example_{i}"
                other_positive_col = f"positive_example_{3-i}"
                sub_dataset["body"] = sub_dataset[body_col]
                sub_dataset["positive_example"] = sub_dataset[other_positive_col]
                sub_dataset["negative_example"] = np.where(
                    np.random.rand(len(sub_dataset)) < 0.5,
                    sub_dataset["negative_example_1"],
                    sub_dataset["negative_example_2"]
                )
                sub_dataset["rule_violation"] = 1

            else:
                body_col = f"negative_example_{i}"
                other_negative_col = f"negative_example_{3-i}"
                sub_dataset["body"] = sub_dataset[body_col]
                sub_dataset["negative_example"] = sub_dataset[other_negative_col]
                sub_dataset["positive_example"] = np.where(
                    np.random.rand(len(sub_dataset)) < 0.5,
                    sub_dataset["positive_example_1"],
                    sub_dataset["positive_example_2"]
                )
                sub_dataset["rule_violation"] = 0

            sub_dataset.drop(columns=["positive_example_1","positive_example_2",
                                      "negative_example_1","negative_example_2"], inplace=True)

            flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(ignore_index=True)

    return dataframe



def build_dataset(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    columns = ["prompt"]
    if "rule_violation" in dataframe:
        dataframe["completion"] = dataframe["rule_violation"].map(
            {
                1: POSITIVE_ANSWER,
                0: NEGATIVE_ANSWER,
            }
        )
        columns.append("completion")

    dataframe = dataframe[columns]
    dataset = Dataset.from_pandas(dataframe)
    dataset.to_pandas().to_csv("/kaggle/working/dataset.csv", index=False)
    return dataset

In [ ]:
%%writefile train.py
import pandas as pd

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers.utils import is_torch_bf16_gpu_available
from utils import build_dataset, get_dataframe_to_train
from constants import DATA_PATH, BASE_MODEL_PATH_1_5B, LORA_PATH_1_5B


def main():
    dataframe = get_dataframe_to_train(DATA_PATH, exclude_legal_advertising=True)
    print(f"Training 1.5B model on {len(dataframe)} samples (excluding legal/advertising rules)")
    train_dataset = build_dataset(dataframe)
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    
    training_args = SFTConfig(
        num_train_epochs=1,
        
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        
        optim="paged_adamw_8bit",
        learning_rate=1e-4,
        weight_decay=0.01,
        max_grad_norm=1.0,
        
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        
        fp16=True,
        dataloader_pin_memory=True,
        
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    
        save_strategy="no",
        report_to="none",
    
        completion_only_loss=True,
        packing=False,
        remove_unused_columns=False,
    )
    
    trainer = SFTTrainer(
        BASE_MODEL_PATH_1_5B,
        args=training_args,
        train_dataset=train_dataset,
        peft_config=lora_config,
    )
    
    trainer.train()
    trainer.save_model(LORA_PATH_1_5B)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile inference_1_5b.py
import os
os.environ["VLLM_USE_V1"] = "0"

import vllm
import torch
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest
from utils import build_dataset
from constants import BASE_MODEL_PATH_1_5B, LORA_PATH_1_5B, DATA_PATH, POSITIVE_ANSWER, NEGATIVE_ANSWER, is_legal_or_advertising_rule
import random
import multiprocessing as mp


def run_inference_on_device(df_slice):
    llm = vllm.LLM(
        BASE_MODEL_PATH_1_5B,
        quantization="gptq",
        tensor_parallel_size=1,
        gpu_memory_utilization=0.85,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=2836,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=64,
        max_num_seqs=32//2,
        max_num_batched_tokens=8192//2,
    )

    tokenizer = llm.get_tokenizer()
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[POSITIVE_ANSWER, NEGATIVE_ANSWER])

    test_dataset = build_dataset(df_slice)
    texts = test_dataset["prompt"]

    outputs = llm.generate(
        texts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH_1_5B)
    )

    log_probs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    predictions = pd.DataFrame(log_probs)[[POSITIVE_ANSWER, NEGATIVE_ANSWER]]
    predictions["row_id"] = df_slice["row_id"].values
    return predictions


def worker(device_id, df_slice, return_dict):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    print(f"[Worker {device_id}] Running on GPU {device_id}, data size={len(df_slice)}")

    preds = run_inference_on_device(df_slice)
    return_dict[device_id] = preds


def main():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    
    test_dataframe_non_legal = test_dataframe[~test_dataframe['rule'].apply(is_legal_or_advertising_rule)].copy()
    print(f"1.5B model will infer on {len(test_dataframe_non_legal)} samples (non-legal/advertising rules)")

    test_dataframe_non_legal["positive_example"] = test_dataframe_non_legal.apply(
        lambda row: random.choice([row["positive_example_1"], row["positive_example_2"]]),
        axis=1
    )
    test_dataframe_non_legal["negative_example"] = test_dataframe_non_legal.apply(
        lambda row: random.choice([row["negative_example_1"], row["negative_example_2"]]),
        axis=1
    )
    test_dataframe_non_legal = test_dataframe_non_legal.drop(
        columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"],
        errors="ignore"
    )

    mid = len(test_dataframe_non_legal) // 2
    df0 = test_dataframe_non_legal.iloc[:mid].reset_index(drop=True)
    df1 = test_dataframe_non_legal.iloc[mid:].reset_index(drop=True)

    manager = mp.Manager()
    return_dict = manager.dict()

    p0 = mp.Process(target=worker, args=(0, df0, return_dict))
    p1 = mp.Process(target=worker, args=(1, df1, return_dict))
    p0.start()
    p1.start()
    p0.join()
    p1.join()

    predictions = pd.concat([return_dict[0], return_dict[1]], ignore_index=True)

    submission = predictions[["row_id", POSITIVE_ANSWER]].rename(columns={POSITIVE_ANSWER: "rule_violation"})
    submission.to_csv("/kaggle/working/submission_1_5b.csv", index=False)
    print(f"Saved submission_1_5b.csv with {len(submission)} predictions")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile inference_32b.py
import os
os.environ["VLLM_USE_V1"] = "0"

import vllm
import torch
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest
from scipy.special import softmax
from constants import BASE_MODEL_PATH_32B, LORA_PATH_32B, DATA_PATH, POSITIVE_ANSWER, NEGATIVE_ANSWER, is_legal_or_advertising_rule
import random


def main():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    
    test_dataframe_legal = test_dataframe[test_dataframe['rule'].apply(is_legal_or_advertising_rule)].copy()
    print(f"32B model will infer on {len(test_dataframe_legal)} samples (legal/advertising rules)")

    llm = vllm.LLM(
        BASE_MODEL_PATH_32B,
        quantization='gptq',
        tensor_parallel_size=torch.cuda.device_count(),
        gpu_memory_utilization=0.95,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=3000,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
    )
    
    tokenizer = llm.get_tokenizer()
    SYS_PROMPT = """
You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
"""
    
    prompts = []
    for i, row in test_dataframe_legal.iterrows():
        pos_ex = random.choice([row['positive_example_1'], row['positive_example_2']])
        neg_ex_1 = random.choice([row['negative_example_1'], row['negative_example_2']])
        
        neg_ex_2 = row['negative_example_2'] if neg_ex_1 == row['negative_example_1'] else row['negative_example_1']
        pos_ex_2 = row['positive_example_2'] if pos_ex == row['positive_example_1'] else row['positive_example_1']
        
        text = f"""
r/{row.subreddit}
Rule: {row.rule}

1) {pos_ex}
Violation: Yes

2) {pos_ex_2}
Violation: Yes

3) {neg_ex_1}
Violation: No

4) {neg_ex_2}
Violation: No

5) {row.body}
"""
        
        messages = [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": text}
        ]
    
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + "Answer:"
        prompts.append(prompt)
    
    test_dataframe_legal["prompt"] = prompts
    
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])
    outputs = llm.generate(
        prompts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH_32B)
    )
    
    logprobs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    logit_matrix = pd.DataFrame(logprobs)[['Yes','No']]
    test_dataframe_legal = pd.concat([test_dataframe_legal.reset_index(drop=True), logit_matrix], axis=1)
    
    test_dataframe_legal[['Yes',"No"]] = test_dataframe_legal[['Yes',"No"]].apply(lambda x: softmax(x.values), axis=1, result_type="expand")
    test_dataframe_legal["rule_violation"] = test_dataframe_legal["Yes"]
    
    submission = test_dataframe_legal[['row_id', 'rule_violation']]
    submission.to_csv("/kaggle/working/submission_32b.csv", index=False)
    print(f"Saved submission_32b.csv with {len(submission)} predictions")
    
    if torch.distributed.is_initialized():
        torch.distributed.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  train_batch_size: 64
  train_micro_batch_size_per_gpu: 4
  
  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false
  
  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5
  
  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false
  
  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1
  
distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

In [ ]:
%%writefile mix_submissions.py
import pandas as pd

def main():
    sub_1_5b = pd.read_csv("/kaggle/working/submission_1_5b.csv")
    sub_32b = pd.read_csv("/kaggle/working/submission_32b.csv")
    
    print(f"1.5B predictions: {len(sub_1_5b)}")
    print(f"32B predictions: {len(sub_32b)}")
    
    final_submission = pd.concat([sub_1_5b, sub_32b], axis=0).sort_values('row_id').reset_index(drop=True)
    
    print(f"Final submission: {len(final_submission)} rows")
    
    rq = final_submission['rule_violation'].rank(method='average') / (len(final_submission) + 1)
    final_submission['rule_violation'] = rq
    
    final_submission.to_csv("/kaggle/working/submission.csv", index=False)
    print("Saved final submission.csv")


if __name__ == "__main__":
    main()

In [ ]:
!python inference.py

In [ ]:
!accelerate launch --config_file accelerate_config.yaml train.py

In [ ]:
!head /kaggle/working/submission.csv

In [ ]:
!python inference_32b.py

In [ ]:
!python inference_1_5b.py

In [ ]:
import pandas as pd
pd.read_csv('/kaggle/working/submission.csv')